# LeetCode 815 — Bus Routes bằng Python

- Link: https://leetcode.com/problems/bus-routes/
- Độ khó: **Hard**
- Chủ đề: BFS, Mô hình hoá đồ thị (điểm dừng / tuyến xe), Đồ thị hai phía, Bảng băm `stop → routes`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numBusesToDestination(self, routes: List[List[int]], source: int, target: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Mỗi tuyến xe buýt `routes[i]` chạy vòng tròn qua một danh sách điểm dừng. Bạn đứng ở `source`, muốn tới `target`, và chỉ được đi bằng xe buýt. Hỏi cần **lên ít nhất bao nhiêu xe**?

Điểm mấu chốt của bài: **cái gì là đỉnh, cái gì là cạnh?** Nếu chọn điểm dừng làm đỉnh và nối mọi cặp điểm dừng trên cùng tuyến, đồ thị có thể có tới `10¹⁰` cạnh. Chọn đúng mô hình (đánh dấu **tuyến đã đi**, hoặc lấy **tuyến** làm đỉnh) đưa độ phức tạp về tuyến tính theo tổng độ dài các tuyến.

Sau notebook này bạn sẽ:

1. Viết BFS "ngây thơ" trên đồ thị điểm dừng (mỗi cặp điểm cùng tuyến là một cạnh).
2. Hiểu vì sao "đã đi tuyến này rồi thì không cần đi lại" là chìa khoá tối ưu.
3. Cài BFS trên **điểm dừng** với tập **tuyến đã dùng**, và BFS trên **tuyến**.
4. Mô hình hoá thành **đồ thị hai phía** (điểm dừng — tuyến) và đọc đáp án = (độ dài đường đi)/2.
5. Xử lý biên: `source == target`, `source` không nằm trên tuyến nào.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `routes`, `routes[i]` là danh sách các điểm dừng mà xe buýt thứ `i` đi qua (lặp lại mãi theo vòng). Bắt đầu ở điểm dừng `source` (chưa ở trên xe), bạn muốn tới `target`. Trả về **số xe buýt ít nhất** phải lên; nếu không thể, trả `-1`.

Lên một xe = được tới **mọi** điểm dừng của tuyến đó (vì xe chạy vòng). Đổi xe chỉ được ở điểm dừng chung.

**Ví dụ 1**

```text
Input: routes = [[1,2,7],[3,6,7]], source = 1, target = 6
Output: 2
Giải thích: đi xe 0 tới điểm 7, đổi sang xe 1 tới điểm 6.
```

**Ví dụ 2**

```text
Input: routes = [[7,12],[4,5,15],[6],[15,19],[9,12,13]], source = 15, target = 12
Output: -1
```

**Ràng buộc**

- `1 <= routes.length <= 500`
- `1 <= routes[i].length <= 10⁵`
- Các giá trị trong mỗi `routes[i]` đôi một khác nhau.
- `sum(routes[i].length) <= 10⁵`
- `0 <= routes[i][j] < 10⁶`
- `0 <= source, target < 10⁶`

In [1]:
from typing import List

EXAMPLES = [
    ([[1, 2, 7], [3, 6, 7]], 1, 6, 2),
    ([[7, 12], [4, 5, 15], [6], [15, 19], [9, 12, 13]], 15, 12, -1),
    ([[1, 2, 3]], 2, 2, 0),             # source == target
    ([[1, 2, 3]], 1, 3, 1),
    ([[1, 2], [3, 4]], 9, 9, 0),        # đứng yên dù 9 không nằm trên tuyến nào
]
for r, s, t, exp in EXAMPLES:
    print(r, s, "->", t, ": kỳ vọng", exp)

[[1, 2, 7], [3, 6, 7]] 1 -> 6 : kỳ vọng 2
[[7, 12], [4, 5, 15], [6], [15, 19], [9, 12, 13]] 15 -> 12 : kỳ vọng -1
[[1, 2, 3]] 2 -> 2 : kỳ vọng 0
[[1, 2, 3]] 1 -> 3 : kỳ vọng 1
[[1, 2], [3, 4]] 9 -> 9 : kỳ vọng 0


## 3. Kiến thức nền

### 3.1. BFS cho đường đi ít cạnh nhất

BFS duyệt theo lớp: lớp `d` gồm các đỉnh cách nguồn đúng `d` cạnh. Lần đầu chạm đích là khoảng cách ngắn nhất. Đánh dấu `visited` khi **đẩy vào** hàng đợi.

In [2]:
from collections import deque, defaultdict

def bfs(adj, src):
    dist = {src: 0}
    q = deque([src])
    while q:
        u = q.popleft()
        for v in adj.get(u, ()):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

print(bfs({1: [2, 3], 2: [4], 3: [4], 4: []}, 1))

{1: 0, 2: 1, 3: 1, 4: 2}


### 3.2. Bảng băm "điểm dừng → các tuyến đi qua"

Đây là cấu trúc dữ liệu trung tâm của bài: `stop_to_routes[s]` = danh sách chỉ số tuyến có điểm `s`. Dựng trong `O(Σ |routes[i]|)`.

In [3]:
def build_stop_to_routes(routes):
    m = defaultdict(list)
    for i, route in enumerate(routes):
        for s in route:
            m[s].append(i)
    return m

print(dict(build_stop_to_routes([[1, 2, 7], [3, 6, 7]])))

{1: [0], 2: [0], 7: [0, 1], 3: [1], 6: [1]}


### 3.3. BFS theo lớp (đếm số lớp)

Thay vì lưu `dist` cho từng đỉnh, có thể xử lý **cả lớp một lúc** và tăng bộ đếm sau mỗi lớp — rất tiện khi "một bước" = "lên một xe".

In [4]:
def bfs_layers(adj, src, dst):
    if src == dst:
        return 0
    seen, layer, d = {src}, [src], 0
    while layer:
        d += 1
        nxt = []
        for u in layer:
            for v in adj.get(u, ()):
                if v == dst:
                    return d
                if v not in seen:
                    seen.add(v)
                    nxt.append(v)
        layer = nxt
    return -1

print(bfs_layers({1: [2, 3], 2: [4], 3: [4], 4: []}, 1, 4))

2


### 3.4. Giao của hai tập hợp

`set(a) & set(b)` hoặc `not set(a).isdisjoint(b)` — dùng để kiểm tra hai tuyến có điểm dừng chung (nối hai tuyến trong đồ thị tuyến).

In [5]:
a, b = {1, 2, 7}, {3, 6, 7}
print(a & b, not a.isdisjoint(b))

{7} True


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Hai cách nhìn

- **Đồ thị điểm dừng:** đỉnh = điểm dừng; hai điểm nối nhau nếu cùng một tuyến. Số xe = số cạnh trên đường đi. Vấn đề: một tuyến `L` điểm tạo `L(L−1)/2` cạnh → tuyến dài `10⁵` điểm cho `5·10⁹` cạnh!
- **Đồ thị tuyến:** đỉnh = tuyến; hai tuyến nối nhau nếu có điểm chung. Nguồn = mọi tuyến chứa `source`; đích = mọi tuyến chứa `target`. Số xe = số đỉnh trên đường đi. Có tối đa 500 đỉnh.

### 4.2. Tính tay ví dụ 1

`routes = [[1,2,7],[3,6,7]]`, `source = 1`, `target = 6`.

- Tuyến chứa 1: `{0}` → lên xe 0 (1 xe), tới được `{1, 2, 7}`.
- Từ điểm 7: tuyến chứa 7 là `{0, 1}`; tuyến 1 chưa dùng → lên xe 1 (2 xe), tới `{3, 6, 7}` — có 6. → **2**.

### 4.3. Tính tay ví dụ 2

`source = 15`: tuyến `{1, 3}` → tới `{4, 5, 15, 19}`. Từ các điểm này không có tuyến mới nào (4, 5, 19 chỉ thuộc tuyến 1 hoặc 3). `target = 12` thuộc tuyến `{0, 4}` — không với tới → **−1**.

### 4.4. Quan sát quan trọng

Nếu đã lên tuyến `r` ở lớp `d`, thì mọi điểm của `r` đều đạt được với `d` xe. Lên lại `r` ở lớp sau **không bao giờ** có ích. → Mỗi tuyến chỉ cần "mở rộng" **một lần**.

In [6]:
routes = [[1, 2, 7], [3, 6, 7]]
s2r = build_stop_to_routes(routes)
print("tuyến chứa source=1:", s2r[1], "| tuyến chứa 7:", s2r[7], "| tuyến chứa target=6:", s2r[6])

tuyến chứa source=1: [0] | tuyến chứa 7: [0, 1] | tuyến chứa target=6: [1]


## 5. Cách 1 — Brute Force: BFS trên đồ thị điểm dừng đầy đủ

**Ý tưởng.** Dựng tường minh đồ thị điểm dừng: với mỗi tuyến, nối mọi cặp điểm dừng trong tuyến. BFS từ `source`, số cạnh tới `target` = số xe.

In [7]:
def buses_brute(routes, source, target):
    if source == target:
        return 0
    adj = defaultdict(set)
    for route in routes:
        for u in route:
            for v in route:
                if u != v:
                    adj[u].add(v)
    return bfs_layers(adj, source, target)

for r, s, t, exp in EXAMPLES:
    print(buses_brute(r, s, t), "(kỳ vọng", exp, ")")
    assert buses_brute(r, s, t) == exp

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp.** Dựng đồ thị `O(Σ Lᵢ²)` — với một tuyến `10⁵` điểm là `10¹⁰` → không thể. Chỉ dùng để đối chiếu với dữ liệu nhỏ.

**Nút thắt.** Ta lưu và duyệt cùng một "cụm" cạnh (cả tuyến) nhiều lần. Thay vì nối từng cặp điểm, hãy coi tuyến như **một siêu đỉnh** — hoặc chỉ cần **đánh dấu tuyến đã dùng**.

## 6. Bước đột phá

**Insight 1 — Mỗi tuyến chỉ mở rộng một lần.** BFS trên điểm dừng, nhưng thay vì duyệt cạnh từng cặp, khi ở điểm `s` ta duyệt các tuyến `r ∈ stop_to_routes[s]` **chưa dùng**, đánh dấu `r` đã dùng, rồi thêm mọi điểm của `r` vào lớp sau. Mỗi tuyến được duyệt `O(Lᵣ)` đúng một lần → tổng `O(Σ Lᵢ)`.

**Insight 2 — Đồ thị tuyến.** Lấy tuyến làm đỉnh, BFS từ các tuyến chứa `source`; dừng khi gặp tuyến chứa `target`. Số đỉnh `≤ 500`.

**Insight 3 — Đồ thị hai phía.** Đỉnh gồm điểm dừng **và** tuyến; cạnh nối điểm `s` với tuyến `r` nếu `s ∈ r`. Đường đi `source → r₁ → s₁ → r₂ → ... → target` có độ dài `2k` với `k` = số xe. BFS chuẩn trên đồ thị này, đáp án `= dist / 2`.

**Quy tắc biên:**
- *Nếu* `source == target` *thì* `0` (không cần lên xe, kể cả khi điểm không thuộc tuyến nào).
- *Nếu* `source` hoặc `target` không thuộc tuyến nào (và khác nhau) *thì* `−1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS trên điểm dừng + đánh dấu tuyến đã dùng

In [8]:
def buses_stops(routes, source, target):
    if source == target:
        return 0
    s2r = build_stop_to_routes(routes)
    if source not in s2r or target not in s2r:
        return -1
    used_route = [False] * len(routes)
    seen_stop = {source}
    layer = [source]
    buses = 0
    while layer:
        buses += 1
        nxt = []
        for s in layer:
            for r in s2r[s]:
                if used_route[r]:
                    continue
                used_route[r] = True
                for t in routes[r]:
                    if t == target:
                        return buses
                    if t not in seen_stop:
                        seen_stop.add(t)
                        nxt.append(t)
        layer = nxt
    return -1

for r, s, t, exp in EXAMPLES:
    assert buses_stops(r, s, t) == exp
print("BFS điểm dừng + tuyến đã dùng: OK")

BFS điểm dừng + tuyến đã dùng: OK


Code nộp LeetCode cho cách 2:

In [9]:
class SolutionStops:
    def numBusesToDestination(self, routes: List[List[int]], source: int, target: int) -> int:
        if source == target:
            return 0
        s2r = defaultdict(list)
        for i, route in enumerate(routes):
            for s in route:
                s2r[s].append(i)
        if source not in s2r or target not in s2r:
            return -1
        used = [False] * len(routes)
        seen = {source}
        q = deque([(source, 0)])
        while q:
            s, d = q.popleft()
            for r in s2r[s]:
                if used[r]:
                    continue
                used[r] = True
                for t in routes[r]:
                    if t == target:
                        return d + 1
                    if t not in seen:
                        seen.add(t)
                        q.append((t, d + 1))
        return -1

for r, s, t, exp in EXAMPLES:
    assert SolutionStops().numBusesToDestination(r, s, t) == exp
print("SolutionStops: OK")

SolutionStops: OK


**Độ phức tạp cách 2:** dựng `s2r` `O(Σ Lᵢ)`; mỗi tuyến duyệt một lần `O(Lᵣ)`; mỗi điểm vào hàng đợi một lần, duyệt `s2r[s]` → tổng `O(Σ Lᵢ)`. Bộ nhớ `O(Σ Lᵢ)`.

### 7.2. Cách 3 — BFS trên đồ thị tuyến

1. Dựng `s2r`.
2. Hai tuyến kề nhau nếu có chung điểm: với mỗi điểm `s`, mọi cặp tuyến trong `s2r[s]` kề nhau. (Hoặc dùng giao tập hợp.)
3. BFS từ tất cả tuyến chứa `source` (khoảng cách 1); dừng khi gặp tuyến chứa `target`.

Lưu ý: nếu một điểm có rất nhiều tuyến (tới 500), số cặp là `500²/2` — vẫn chấp nhận được; tổng số cạnh của đồ thị tuyến tối đa `C(500, 2) ≈ 1.25·10⁵`.

In [10]:
def buses_routes(routes, source, target):
    if source == target:
        return 0
    s2r = build_stop_to_routes(routes)
    if source not in s2r or target not in s2r:
        return -1
    R = len(routes)
    adj = [set() for _ in range(R)]
    for rs in s2r.values():
        for i in range(len(rs)):
            for j in range(i + 1, len(rs)):
                adj[rs[i]].add(rs[j])
                adj[rs[j]].add(rs[i])
    goal = set(s2r[target])
    dist = {r: 1 for r in s2r[source]}
    q = deque(s2r[source])
    while q:
        r = q.popleft()
        if r in goal:
            return dist[r]
        for nr in adj[r]:
            if nr not in dist:
                dist[nr] = dist[r] + 1
                q.append(nr)
    return -1

for r, s, t, exp in EXAMPLES:
    assert buses_routes(r, s, t) == exp
print("BFS đồ thị tuyến: OK")

BFS đồ thị tuyến: OK


Code nộp LeetCode cho cách 3 (dùng giao tập hợp để dựng cạnh — đơn giản, `O(R² · L)` xấu nhất nhưng `R ≤ 500`; `isdisjoint` dừng sớm khi tìm thấy phần tử chung):

In [11]:
class SolutionRoutes:
    def numBusesToDestination(self, routes: List[List[int]], source: int, target: int) -> int:
        if source == target:
            return 0
        sets = [set(r) for r in routes]
        R = len(routes)
        adj = [[] for _ in range(R)]
        for i in range(R):
            for j in range(i + 1, R):
                if not sets[i].isdisjoint(sets[j]):
                    adj[i].append(j)
                    adj[j].append(i)
        start = [i for i in range(R) if source in sets[i]]
        goal = {i for i in range(R) if target in sets[i]}
        dist = {i: 1 for i in start}
        q = deque(start)
        while q:
            r = q.popleft()
            if r in goal:
                return dist[r]
            for nr in adj[r]:
                if nr not in dist:
                    dist[nr] = dist[r] + 1
                    q.append(nr)
        return -1

for r, s, t, exp in EXAMPLES:
    assert SolutionRoutes().numBusesToDestination(r, s, t) == exp
print("SolutionRoutes: OK")

SolutionRoutes: OK


**Độ phức tạp cách 3:** bản dùng `s2r`: `O(Σ Lᵢ + Σ_s |s2r[s]|²)`; bản giao tập hợp: `O(R² · min L)` xấu nhất. BFS `O(R + E)` với `E ≤ R²/2`. Bộ nhớ `O(Σ Lᵢ + R²)`.

### 7.3. Cách 4 — BFS trên đồ thị hai phía (điểm dừng ∪ tuyến)

Đánh số đỉnh: tuyến `r` là `('R', r)`, điểm `s` là `('S', s)`. Cạnh `('S', s) — ('R', r)` nếu `s ∈ routes[r]`. BFS từ `('S', source)`; khoảng cách tới `('S', target)` luôn chẵn `= 2·(số xe)`.

In [12]:
def buses_bipartite(routes, source, target):
    if source == target:
        return 0
    s2r = build_stop_to_routes(routes)
    start, goal = ('S', source), ('S', target)

    def neighbors(node):
        kind, x = node
        if kind == 'S':
            return [('R', r) for r in s2r.get(x, ())]
        return [('S', s) for s in routes[x]]

    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        if u == goal:
            return dist[u] // 2
        for v in neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return -1

for r, s, t, exp in EXAMPLES:
    assert buses_bipartite(r, s, t) == exp
print("BFS đồ thị hai phía: OK")

BFS đồ thị hai phía: OK


In [13]:
class SolutionBipartite:
    def numBusesToDestination(self, routes: List[List[int]], source: int, target: int) -> int:
        if source == target:
            return 0
        s2r = defaultdict(list)
        for i, route in enumerate(routes):
            for s in route:
                s2r[s].append(i)
        R = len(routes)
        # đỉnh tuyến: 0..R-1 ; đỉnh điểm dừng: R + s (dùng dict để khỏi cấp phát 10^6 ô)
        dist = {R + source: 0}
        q = deque([R + source])
        while q:
            u = q.popleft()
            d = dist[u]
            if u >= R:                       # điểm dừng
                s = u - R
                if s == target:
                    return d // 2
                nbrs = s2r.get(s, ())
            else:                            # tuyến
                nbrs = [R + s for s in routes[u]]
            for v in nbrs:
                if v not in dist:
                    dist[v] = d + 1
                    q.append(v)
        return -1

for r, s, t, exp in EXAMPLES:
    assert SolutionBipartite().numBusesToDestination(r, s, t) == exp
print("SolutionBipartite: OK")

SolutionBipartite: OK


**Độ phức tạp cách 4:** số đỉnh `R + (số điểm khác nhau)`, số cạnh `Σ Lᵢ` (mỗi cặp điểm–tuyến một cạnh, duyệt hai chiều) → **`O(Σ Lᵢ)`**. Đây thực chất là cách 2 viết theo ngôn ngữ đồ thị: "đánh dấu tuyến đã dùng" = "đỉnh tuyến đã thăm".

## 8. TRACE từng bước

Trace cách 2 trên một ví dụ có 3 lần đổi xe:

```text
routes = [[1, 2, 3], [3, 4, 5], [5, 6, 7], [7, 8]]   source = 1, target = 8
```

In [14]:
def trace_stops(routes, source, target):
    if source == target:
        print("source == target -> 0"); return 0
    s2r = build_stop_to_routes(routes)
    used = [False] * len(routes)
    seen = {source}
    layer = [source]
    buses = 0
    while layer:
        buses += 1
        nxt = []
        print(f"--- lớp {buses} (lên xe thứ {buses}), xuất phát từ các điểm {layer}")
        for s in layer:
            for r in s2r.get(s, ()):
                if used[r]:
                    print(f"   điểm {s}: tuyến {r} đã dùng, bỏ qua")
                    continue
                used[r] = True
                print(f"   điểm {s}: lên tuyến {r} = {routes[r]}")
                for t in routes[r]:
                    if t == target:
                        print(f"   -> tới target {target} sau {buses} xe")
                        return buses
                    if t not in seen:
                        seen.add(t); nxt.append(t)
        layer = nxt
    print("không tới được -> -1")
    return -1

trace_stops([[1, 2, 3], [3, 4, 5], [5, 6, 7], [7, 8]], 1, 8)

--- lớp 1 (lên xe thứ 1), xuất phát từ các điểm [1]
   điểm 1: lên tuyến 0 = [1, 2, 3]
--- lớp 2 (lên xe thứ 2), xuất phát từ các điểm [2, 3]
   điểm 2: tuyến 0 đã dùng, bỏ qua
   điểm 3: tuyến 0 đã dùng, bỏ qua
   điểm 3: lên tuyến 1 = [3, 4, 5]
--- lớp 3 (lên xe thứ 3), xuất phát từ các điểm [4, 5]
   điểm 4: tuyến 1 đã dùng, bỏ qua
   điểm 5: tuyến 1 đã dùng, bỏ qua
   điểm 5: lên tuyến 2 = [5, 6, 7]
--- lớp 4 (lên xe thứ 4), xuất phát từ các điểm [6, 7]
   điểm 6: tuyến 2 đã dùng, bỏ qua
   điểm 7: tuyến 2 đã dùng, bỏ qua
   điểm 7: lên tuyến 3 = [7, 8]
   -> tới target 8 sau 4 xe


4

Bảng trạng thái:

| lớp (số xe) | điểm xuất phát | tuyến mới lên | điểm mới đạt được |
|---|---|---|---|
| 1 | {1} | 0 | {2, 3} |
| 2 | {2, 3} | 1 (tại 3) | {4, 5} |
| 3 | {4, 5} | 2 (tại 5) | {6, 7} |
| 4 | {6, 7} | 3 (tại 7) | 8 = target ✔ |

Trace cách 4 (hai phía) trên ví dụ 1 — in thứ tự thăm đỉnh và khoảng cách:

In [15]:
def trace_bipartite(routes, source, target):
    s2r = build_stop_to_routes(routes)
    start, goal = ('S', source), ('S', target)
    dist = {start: 0}
    q = deque([start])
    order = []
    while q:
        u = q.popleft()
        order.append((u, dist[u]))
        if u == goal:
            break
        kind, x = u
        nbrs = [('R', r) for r in s2r.get(x, ())] if kind == 'S' else [('S', s) for s in routes[x]]
        for v in nbrs:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    for node, d in order:
        print(f"thăm {node[0]}{node[1]:<3} dist = {d}")
    print("số xe =", dist.get(goal, -2) // 2 if goal in dist else -1)

trace_bipartite([[1, 2, 7], [3, 6, 7]], 1, 6)

thăm S1   dist = 0
thăm R0   dist = 1
thăm S2   dist = 2
thăm S7   dist = 2
thăm R1   dist = 3
thăm S3   dist = 4
thăm S6   dist = 4
số xe = 2


## 9. Code nộp LeetCode

In [16]:
from collections import defaultdict, deque
from typing import List

class Solution:
    def numBusesToDestination(self, routes: List[List[int]], source: int, target: int) -> int:
        if source == target:
            return 0
        stop_to_routes = defaultdict(list)
        for i, route in enumerate(routes):
            for s in route:
                stop_to_routes[s].append(i)
        if source not in stop_to_routes or target not in stop_to_routes:
            return -1
        used_route = [False] * len(routes)
        seen_stop = {source}
        layer = [source]
        buses = 0
        while layer:
            buses += 1                       # lên thêm một xe
            nxt = []
            for s in layer:
                for r in stop_to_routes[s]:
                    if used_route[r]:
                        continue
                    used_route[r] = True     # mỗi tuyến chỉ mở rộng một lần
                    for t in routes[r]:
                        if t == target:
                            return buses
                        if t not in seen_stop:
                            seen_stop.add(t)
                            nxt.append(t)
            layer = nxt
        return -1

for r, s, t, exp in EXAMPLES:
    assert Solution().numBusesToDestination(r, s, t) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Mô hình.** Gọi `G` là đồ thị hai phía điểm–tuyến. Một hành trình lên `k` xe tương ứng đường đi `source = s₀, r₁, s₁, r₂, ..., r_k, s_k = target` trong `G` (lên tuyến `rᵢ` tại `s_{i−1}`, xuống tại `sᵢ`), độ dài `2k`. Ngược lại mọi đường đi trong `G` từ `source` tới `target` cho một hành trình. Vậy số xe tối thiểu = `dist_G(source, target) / 2` → **cách 4** đúng (BFS cho khoảng cách ngắn nhất trên đồ thị không trọng số).

**Cách 2** là BFS trên `G` được "nén lớp": mỗi vòng `while` đi qua một đỉnh tuyến và một đỉnh điểm (2 cạnh). `used_route` = đã thăm đỉnh tuyến, `seen_stop` = đã thăm đỉnh điểm. Theo tính chất BFS, lớp thứ `b` chứa đúng các điểm dừng có khoảng cách `2b`, tức đạt được với `b` xe mà không ít hơn. Lần đầu gặp `target` → đáp án.

**Cách 3** chiếu `G` lên phía tuyến: hai tuyến kề nhau ⇔ có điểm chung ⇔ trong `G` cách nhau 2. BFS đa nguồn từ các tuyến chứa `source` (khoảng cách 1) cho số tuyến tối thiểu tới một tuyến chứa `target`.

**Cách 1** dựng đồ thị điểm–điểm: `u, v` kề nhau ⇔ cách nhau 2 trong `G` qua một tuyến. Khoảng cách điểm–điểm = số xe.

**Biên:** `source == target` → 0 xe (đường đi độ dài 0). Nếu không có đường → `−1`.

## 11. Phân tích Time/Space

Gọi `N = Σ |routes[i]|` (≤ 10⁵), `R = len(routes)` (≤ 500).

| Cách | Time | Space |
|---|---|---|
| 1. Đồ thị điểm đầy đủ | `O(Σ Lᵢ²)` | `O(Σ Lᵢ²)` |
| 2. BFS điểm + tuyến đã dùng | `O(N)` | `O(N)` |
| 3. BFS đồ thị tuyến | `O(N + R²)` (bản `s2r`: `O(N + Σ_s |s2r[s]|²)`) | `O(N + R²)` |
| 4. BFS hai phía | `O(N)` | `O(N)` |

In [17]:
import random, time

def random_routes(R, L, V, rnd):
    return [rnd.sample(range(V), min(V, rnd.randint(1, L))) for _ in range(R)]

rnd = random.Random(815)
# Test lớn: 500 tuyến, tổng ~1e5 điểm
pool = rnd.sample(range(10**6), 500 * 200)    # các điểm đôi một khác nhau
big = [pool[i * 200:(i + 1) * 200] for i in range(500)]
for i in range(1, 500):                       # nối chuỗi: tuyến i chung đúng 1 điểm với tuyến i-1
    big[i][0] = big[i - 1][-1]
big = [list(dict.fromkeys(r)) for r in big]
src, dst = big[0][1], big[-1][-1]
for name, fn in [("stops", buses_stops), ("routes", buses_routes), ("bipartite", buses_bipartite),
                 ("Solution", lambda a, b, c: Solution().numBusesToDestination(a, b, c))]:
    t0 = time.perf_counter()
    res = fn(big, src, dst)
    print(f"{name:9}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

stops    : 500  (85 ms)


routes   : 500  (80 ms)


bipartite: 500  (137 ms)


Solution : 500  (107 ms)


## 12. Bộ test

In [18]:
methods = {
    "brute": buses_brute,
    "stops": buses_stops,
    "routes": buses_routes,
    "bipartite": buses_bipartite,
    "SolutionStops": lambda a, b, c: SolutionStops().numBusesToDestination(a, b, c),
    "SolutionRoutes": lambda a, b, c: SolutionRoutes().numBusesToDestination(a, b, c),
    "SolutionBipartite": lambda a, b, c: SolutionBipartite().numBusesToDestination(a, b, c),
    "Solution": lambda a, b, c: Solution().numBusesToDestination(a, b, c),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([[1, 7], [3, 5]], 5, 5, 0),
    ([[1, 2]], 1, 3, -1),                         # target không thuộc tuyến nào
    ([[1, 2]], 3, 1, -1),                         # source không thuộc tuyến nào
    ([[1, 2, 3], [3, 4, 5], [5, 6, 7], [7, 8]], 1, 8, 4),
    ([[1, 2, 3], [3, 4, 5], [5, 6, 7], [7, 8], [1, 8]], 1, 8, 1),   # có tuyến tắt
    ([[2], [2, 8]], 8, 2, 1),
    ([[0, 1, 6, 16, 22, 23], [14, 15, 24, 32], [4, 10, 12, 20, 24, 28, 33],
      [1, 10, 11, 19, 27, 33], [11, 23, 25, 28], [15, 20, 21, 23, 29], [29]], 4, 21, 2),
]
for name, fn in methods.items():
    for r, s, t, exp in edge:
        assert fn(r, s, t) == exp, (name, r, s, t, fn(r, s, t))
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force
for seed in range(1000):
    r_ = random.Random(seed)
    V = r_.randint(2, 15)
    routes = random_routes(r_.randint(1, 6), r_.randint(1, 5), V, r_)
    s, t = r_.randrange(V + 2), r_.randrange(V + 2)     # có thể ngoài mọi tuyến
    ref = buses_brute(routes, s, t)
    for name, fn in methods.items():
        assert fn(routes, s, t) == ref, (name, routes, s, t)
print("1000 bộ ngẫu nhiên: khớp brute force")

# 12.3 Vừa: các cách nhanh khớp nhau
for seed in range(30):
    r_ = random.Random(3000 + seed)
    routes = random_routes(r_.randint(50, 200), 60, 3000, r_)
    s, t = r_.randrange(3000), r_.randrange(3000)
    vals = {name: fn(routes, s, t) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("Dữ liệu vừa: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK
1000 bộ ngẫu nhiên: khớp brute force


Dữ liệu vừa: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không kiểm tra `source == target` trước (trả `-1` thay vì `0`)

**Code sai:**

In [19]:
def wrong_same(routes, source, target):
    s2r = build_stop_to_routes(routes)
    if source not in s2r or target not in s2r:
        return -1                         # SAI: chạy trước kiểm tra source == target
    if source == target:
        return 0
    return buses_stops(routes, source, target)

print("sai:", wrong_same([[1, 2], [3, 4]], 9, 9), " đúng:", Solution().numBusesToDestination([[1, 2], [3, 4]], 9, 9))

sai: -1  đúng: 0


**Vì sao sai:** đứng sẵn ở đích thì không cần xe nào, dù điểm đó không thuộc tuyến nào.

**Code đúng:** `if source == target: return 0` **đầu tiên**.

### Lỗi 2 — Không đánh dấu tuyến đã dùng (duyệt lại tuyến nhiều lần)

**Code sai:**

In [20]:
def wrong_no_route_mark(routes, source, target):
    if source == target:
        return 0
    s2r = build_stop_to_routes(routes)
    seen, layer, buses, work = {source}, [source], 0, 0
    while layer:
        buses += 1
        nxt = []
        for s in layer:
            for r in s2r.get(s, ()):          # SAI (chậm): không đánh dấu tuyến
                for t in routes[r]:
                    work += 1
                    if t == target:
                        return buses, work
                    if t not in seen:
                        seen.add(t); nxt.append(t)
        layer = nxt
    return -1, work

one_long = [list(range(5000))]
print("không đánh dấu tuyến (kết quả, số phép duyệt):", wrong_no_route_mark(one_long + [[4999, 10**6 - 1]], 0, 10**6 - 1))

def count_work_correct(routes, source, target):
    s2r = build_stop_to_routes(routes)
    used = [False] * len(routes); seen, layer, buses, work = {source}, [source], 0, 0
    while layer:
        buses += 1; nxt = []
        for s in layer:
            for r in s2r.get(s, ()):
                if used[r]:
                    continue
                used[r] = True
                for t in routes[r]:
                    work += 1
                    if t == target:
                        return buses, work
                    if t not in seen:
                        seen.add(t); nxt.append(t)
        layer = nxt
    return -1, work

print("có đánh dấu tuyến   (kết quả, số phép duyệt):", count_work_correct(one_long + [[4999, 10**6 - 1]], 0, 10**6 - 1))

không đánh dấu tuyến (kết quả, số phép duyệt): (2, 25000002)
có đánh dấu tuyến   (kết quả, số phép duyệt): (2, 5002)


**Vì sao sai:** kết quả đúng nhưng mỗi điểm trên tuyến dài lại duyệt lại **cả tuyến** → `O(L²)` (ở đây ~25 triệu phép thay vì ~5 nghìn). Với tuyến `10⁵` điểm → TLE.

**Code đúng:** `used_route[r] = True` khi mở rộng lần đầu.

### Lỗi 3 — Đếm số điểm dừng (số cạnh điểm–điểm theo vị trí) thay vì số xe

**Code sai:**

In [21]:
def wrong_count_stops(routes, source, target):
    """Coi mỗi bước đi giữa hai điểm KỀ NHAU trên tuyến là 1 đơn vị."""
    adj = defaultdict(set)
    for route in routes:
        for i in range(len(route)):
            u, v = route[i], route[(i + 1) % len(route)]
            adj[u].add(v); adj[v].add(u)
    return bfs_layers(adj, source, target)

r = [[1, 2, 3, 4, 5, 6, 7]]
print("sai:", wrong_count_stops(r, 1, 4), " đúng:", Solution().numBusesToDestination(r, 1, 4))

sai: 3  đúng: 1


**Vì sao sai:** đề hỏi **số xe**, không phải số trạm đi qua. Trên cùng một tuyến, tới bất kỳ điểm nào cũng chỉ tốn 1 xe.

**Code đúng:** một bước BFS = lên một tuyến mới.

### Lỗi 4 — Cấp phát mảng `visited` theo giá trị điểm dừng (10⁶) cho mỗi test

Không sai về logic nhưng tốn bộ nhớ/thời gian khởi tạo. Dùng `set`/`dict` cho điểm dừng, mảng cho tuyến (`≤ 500`).

## 14. Mẹo & nhận dạng pattern

- **"Ít lần đổi phương tiện / ít nhóm nhất"** → BFS mà mỗi bước là **một nhóm** (tuyến, thang máy, cổng dịch chuyển…), không phải một cạnh lẻ.
- **Nhóm làm siêu đỉnh** hoặc **đánh dấu nhóm đã dùng** để tránh `O(L²)` cạnh.
- **Đồ thị hai phía phần tử–nhóm**: đáp án = khoảng cách / 2.
- **Bảng băm ngược** `phần tử → các nhóm chứa nó` là bước dựng đầu tiên.
- Cùng họ: **1345 Jump Game IV** (nhóm các chỉ số cùng giá trị — xoá nhóm sau khi dùng), **127 Word Ladder** (nhóm theo mẫu `h*t`), **433 Minimum Genetic Mutation**.

## 15. Biến thể

1. **In ra hành trình** (các tuyến đã đi): lưu `parent` của mỗi điểm/tuyến rồi truy ngược.
2. **Mỗi xe có giá vé khác nhau**, tìm tổng tiền nhỏ nhất: đồ thị hai phía có trọng số trên cạnh điểm→tuyến → Dijkstra.
3. **Ít điểm dừng đi qua nhất** (không phải ít xe): đồ thị điểm kề theo thứ tự trên tuyến (vòng).
4. **Xe chạy một chiều, không vòng:** chỉ đi tới các điểm phía sau trên tuyến — đồ thị có hướng.
5. **Nhiều truy vấn `(source, target)`:** tiền xử lý BFS từ mỗi tuyến trên đồ thị tuyến (`R ≤ 500` → `R` lần BFS).

Minh hoạ biến thể 1 — in hành trình:

In [22]:
def bus_itinerary(routes, source, target):
    if source == target:
        return []
    s2r = build_stop_to_routes(routes)
    used = [False] * len(routes)
    parent = {source: None}        # điểm -> (điểm trước, tuyến đã đi)
    q = deque([source])
    while q:
        s = q.popleft()
        for r in s2r.get(s, ()):
            if used[r]:
                continue
            used[r] = True
            for t in routes[r]:
                if t not in parent:
                    parent[t] = (s, r)
                    if t == target:
                        path, cur = [], t
                        while parent[cur]:
                            prev, rr = parent[cur]
                            path.append((prev, rr, cur))
                            cur = prev
                        return path[::-1]
                    q.append(t)
    return None

for leg in bus_itinerary([[1, 2, 3], [3, 4, 5], [5, 6, 7], [7, 8]], 1, 8):
    print(f"tại {leg[0]} lên tuyến {leg[1]}, xuống ở {leg[2]}")

tại 1 lên tuyến 0, xuống ở 3
tại 3 lên tuyến 1, xuống ở 5
tại 5 lên tuyến 2, xuống ở 7
tại 7 lên tuyến 3, xuống ở 8


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đồ thị điểm đầy đủ | nối mọi cặp điểm cùng tuyến | `O(Σ L²)` | `O(Σ L²)` | đối chiếu, dữ liệu nhỏ |
| 2. BFS điểm + tuyến đã dùng | mỗi tuyến mở rộng 1 lần | `O(N)` | `O(N)` | **nộp bài** |
| 3. BFS đồ thị tuyến | tuyến là đỉnh | `O(N + R²)` | `O(N + R²)` | khi `R` nhỏ, nhiều truy vấn |
| 4. BFS hai phía | điểm ∪ tuyến, dist/2 | `O(N)` | `O(N)` | cách nhìn tổng quát, dễ mở rộng (Dijkstra) |

## 17. Bài tập

**Bài 1 (đoán output).** `routes = [[1, 5, 9], [2, 5], [9, 10]]`: số xe từ `1` tới `10`? Từ `2` tới `10`? Từ `10` tới `11`?

**Bài 2 (tính tay).** Với `routes = [[0,1,6,16,22,23],[14,15,24,32],[4,10,12,20,24,28,33],[1,10,11,19,27,33],[11,23,25,28],[15,20,21,23,29],[29]]`, `source = 4`, `target = 21`: liệt kê các lớp BFS (tuyến được lên ở mỗi lớp).

**Bài 3 (điền TODO).** Hoàn thiện `all_pairs_routes(routes)` trả về ma trận `R × R` số xe tối thiểu để đi từ (đang ở trên) tuyến `i` tới tuyến `j` (BFS từ mỗi tuyến trên đồ thị tuyến).

**Bài 4 (tự viết lại).** Viết lại cách 2 với **BFS hai chiều** (từ `source` và `target` luân phiên mở rộng lớp nhỏ hơn).

**Bài 5 (mini challenge).** Biến thể 2: `fare[i]` là giá vé tuyến `i`; tìm tổng tiền nhỏ nhất từ `source` tới `target` (Dijkstra trên đồ thị hai phía).

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def all_pairs_routes(routes):
    # TODO: dựng đồ thị tuyến; BFS từ mỗi tuyến; dist[i][i] = 1 (đang trên tuyến i: 1 xe)
    pass

# Bài 4 — TODO
def buses_bidirectional(routes, source, target):
    # TODO: hai tập tuyến đã dùng, hai lớp điểm; mở rộng phía nhỏ; gặp nhau khi một tuyến được cả hai phía chạm
    pass

# Bài 5 — TODO
def min_fare(routes, fare, source, target):
    # TODO: Dijkstra; cạnh điểm -> tuyến trọng số fare[r], tuyến -> điểm trọng số 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tuyến chứa 1: `{0}`; tuyến chứa 10: `{2}`.
- Mức 2: tuyến 0 và 2 chung điểm 9 → 2 xe. Từ 2: tuyến 1 → (5) → tuyến 0 → (9) → tuyến 2 → 3 xe.
- Mức 3: 11 không thuộc tuyến nào → −1.

**Bài 2.**
- Mức 1: tuyến chứa 4: `{2}`.
- Mức 2: lớp 1 lên tuyến 2, đạt `{10, 12, 20, 24, 28, 33}`.
- Mức 3: lớp 2: từ 20 lên tuyến 5 (chứa 21) → 2 xe.

**Bài 3.**
- Mức 1: đồ thị tuyến như cách 3.
- Mức 2: BFS từ mỗi `i`, `dist[i] = 1`.
- Mức 3: không tới được → `-1`.

**Bài 4.**
- Mức 1: hai phía đều "lên xe" (đếm số tuyến).
- Mức 2: gặp nhau khi một tuyến vừa được phía này mở rộng đã được phía kia dùng → tổng số xe = `bx + by − 1`... cẩn thận đếm tuyến chung một lần.
- Mức 3: kiểm tra bằng brute force trên dữ liệu ngẫu nhiên.

**Bài 5.**
- Mức 1: đỉnh `('S', s)` và `('R', r)`.
- Mức 2: `S → R` tốn `fare[r]`, `R → S` tốn 0.
- Mức 3: dùng `heapq`, dừng khi lấy ra `('S', target)`.

## 19. Đáp án tham khảo

In [24]:
import heapq

S = Solution()
# Bài 1
r1 = [[1, 5, 9], [2, 5], [9, 10]]
assert S.numBusesToDestination(r1, 1, 10) == 2
assert S.numBusesToDestination(r1, 2, 10) == 3
assert S.numBusesToDestination(r1, 10, 11) == -1

# Bài 2
r2 = [[0, 1, 6, 16, 22, 23], [14, 15, 24, 32], [4, 10, 12, 20, 24, 28, 33],
      [1, 10, 11, 19, 27, 33], [11, 23, 25, 28], [15, 20, 21, 23, 29], [29]]
assert S.numBusesToDestination(r2, 4, 21) == 2
trace_stops(r2, 4, 21)

# Bài 3
def all_pairs_routes_ans(routes):
    R = len(routes)
    s2r = build_stop_to_routes(routes)
    adj = [set() for _ in range(R)]
    for rs in s2r.values():
        for a in rs:
            for b in rs:
                if a != b:
                    adj[a].add(b)
    D = [[-1] * R for _ in range(R)]
    for i in range(R):
        D[i][i] = 1
        q = deque([i])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if D[i][v] == -1:
                    D[i][v] = D[i][u] + 1
                    q.append(v)
    return D

D = all_pairs_routes_ans(r1)
assert D == [[1, 2, 2], [2, 1, 3], [2, 3, 1]]

# Bài 4
def buses_bidirectional_ans(routes, source, target):
    if source == target:
        return 0
    s2r = build_stop_to_routes(routes)
    if source not in s2r or target not in s2r:
        return -1
    # dist theo tuyến từ mỗi phía
    dA = {r: 1 for r in s2r[source]}
    dB = {r: 1 for r in s2r[target]}
    common = set(dA) & set(dB)
    if common:
        return 1
    seenA, seenB = {source}, {target}
    frontA, frontB = list(dA), list(dB)
    best = float('inf')
    while frontA and frontB:
        if len(frontA) > len(frontB):
            dA, dB, frontA, frontB, seenA, seenB = dB, dA, frontB, frontA, seenB, seenA
        nxt = []
        for r in frontA:
            for s in routes[r]:
                if s in seenA:
                    continue
                seenA.add(s)
                for nr in s2r[s]:
                    if nr in dB:
                        best = min(best, dA[r] + dB[nr])
                    if nr not in dA:
                        dA[nr] = dA[r] + 1
                        nxt.append(nr)
        if best < float('inf'):
            return best
        frontA = nxt
    return -1

for seed in range(1500):
    r_ = random.Random(seed)
    V = r_.randint(2, 15)
    routes = random_routes(r_.randint(1, 7), r_.randint(1, 5), V, r_)
    s, t = r_.randrange(V + 1), r_.randrange(V + 1)
    assert buses_bidirectional_ans(routes, s, t) == buses_brute(routes, s, t), (routes, s, t)

# Bài 5
def min_fare_ans(routes, fare, source, target):
    if source == target:
        return 0
    s2r = build_stop_to_routes(routes)
    dist = {('S', source): 0}
    pq = [(0, 'S', source)]
    while pq:
        d, kind, x = heapq.heappop(pq)
        if d > dist.get((kind, x), float('inf')):
            continue
        if kind == 'S' and x == target:
            return d
        if kind == 'S':
            nb = [(('R', r), fare[r]) for r in s2r.get(x, ())]
        else:
            nb = [(('S', s), 0) for s in routes[x]]
        for (k2, y), w in nb:
            nd = d + w
            if nd < dist.get((k2, y), float('inf')):
                dist[(k2, y)] = nd
                heapq.heappush(pq, (nd, k2, y))
    return -1

assert min_fare_ans([[1, 2, 7], [3, 6, 7]], [5, 3], 1, 6) == 8
assert min_fare_ans([[1, 6], [1, 2, 7], [3, 6, 7]], [10, 1, 1], 1, 6) == 2   # 2 xe rẻ hơn 1 xe đắt
for seed in range(300):
    r_ = random.Random(seed)
    V = r_.randint(2, 12)
    routes = random_routes(r_.randint(1, 5), 4, V, r_)
    fare = [1] * len(routes)
    s, t = r_.randrange(V), r_.randrange(V)
    assert min_fare_ans(routes, fare, s, t) == buses_brute(routes, s, t)   # giá 1 = đếm số xe
print("Tất cả đáp án đúng")

--- lớp 1 (lên xe thứ 1), xuất phát từ các điểm [4]
   điểm 4: lên tuyến 2 = [4, 10, 12, 20, 24, 28, 33]
--- lớp 2 (lên xe thứ 2), xuất phát từ các điểm [10, 12, 20, 24, 28, 33]
   điểm 10: tuyến 2 đã dùng, bỏ qua
   điểm 10: lên tuyến 3 = [1, 10, 11, 19, 27, 33]
   điểm 12: tuyến 2 đã dùng, bỏ qua
   điểm 20: tuyến 2 đã dùng, bỏ qua
   điểm 20: lên tuyến 5 = [15, 20, 21, 23, 29]
   -> tới target 21 sau 2 xe
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (BFS theo nhóm, đánh dấu nhóm đã dùng):**

```python
if source == target: return 0
s2r = defaultdict(list)
for i, route in enumerate(routes):
    for s in route: s2r[s].append(i)
used = [False]*len(routes); seen = {source}; layer = [source]; buses = 0
while layer:
    buses += 1; nxt = []
    for s in layer:
        for r in s2r[s]:
            if used[r]: continue
            used[r] = True
            for t in routes[r]:
                if t == target: return buses
                if t not in seen: seen.add(t); nxt.append(t)
    layer = nxt
return -1
```

**Checklist trước khi submit:**

- [ ] `source == target` → `0` (kiểm tra **đầu tiên**).
- [ ] Dựng `stop → routes` một lần.
- [ ] Đánh dấu **tuyến** đã dùng (tránh `O(L²)`).
- [ ] Đếm **số xe** (lớp BFS), không phải số điểm dừng.
- [ ] Dùng `set`/`dict` cho điểm dừng (giá trị tới 10⁶).